# 1. Konfigurasi & Load Environment Variables

In [10]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [29]:
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# ─── Konfigurasi utama (ubah sesuai kebutuhan) ───────────────────────────────
MD_FOLDER_PATH     = "/kaggle/input/datasets/brianmaxwellketaren/panduan-skripsi-rag-md"                # 📁 Folder berisi file .md
COLLECTION_NAME    = "rag_docs"              # Nama collection Qdrant
EMBEDDING_MODEL    = "paraphrase-multilingual-MiniLM-L12-v2"
VECTOR_DIM         = 384                     # Dimensi vektor MiniLM-L12-v2

# Semantic chunking
WINDOW_SIZE        = 2
MIN_SENTENCES    = 2 
PERCENTILE_THRESH  = 80                      # breakpoint_threshold (percentile)

# Retrieval
TOP_K              = 25                      # Jumlah vektor yang di-retrieve
TOP_N_RERANK       = 5                       # Jumlah hasil setelah reranking

# Groq
GROQ_MODEL         = "meta-llama/llama-4-scout-17b-16e-instruct"          # Model Groq ~9B parameter
MAX_TOKENS_OUTPUT   = 2048                     # Maks token jawaban LLM
MAX_CHARS_PER_CHUNK = 1500                     # Maks karakter per chunk dalam context
                                               # (~375 token; 5 chunk × 1500 ≈ 1875 token context)

# Credentials dari .env
QDRANT_URL         = user_secrets.get_secret("QDRANT_URL")
QDRANT_API_KEY     = user_secrets.get_secret("QDRANT_API_KEY")
GROQ_API_KEY       = user_secrets.get_secret("GROQ_API_KEY")

# Validasi
assert QDRANT_URL,     "❌ QDRANT_URL tidak ditemukan di .env!"
assert QDRANT_API_KEY, "❌ QDRANT_API_KEY tidak ditemukan di .env!"
assert GROQ_API_KEY,   "❌ GROQ_API_KEY tidak ditemukan di .env!"

print("✅ Konfigurasi berhasil dimuat!")
print(f"   📁 Folder MD          : {MD_FOLDER_PATH}")
print(f"   🗄️  Qdrant URL         : {QDRANT_URL}")
print(f"   🤖 Groq Model         : {GROQ_MODEL}")
print(f"   ✂️  Max chars/chunk    : {MAX_CHARS_PER_CHUNK}")
print(f"   📤 Max tokens output  : {MAX_TOKENS_OUTPUT}")

✅ Konfigurasi berhasil dimuat!
   📁 Folder MD          : /kaggle/input/datasets/brianmaxwellketaren/panduan-skripsi-rag-md
   🗄️  Qdrant URL         : https://ea295c4d-3efb-4676-9964-a20d09b4e162.us-east4-0.gcp.cloud.qdrant.io
   🤖 Groq Model         : meta-llama/llama-4-scout-17b-16e-instruct
   ✂️  Max chars/chunk    : 1500
   📤 Max tokens output  : 1024


# 2. Membaca Semua File `.md` dari Folder

In [12]:
import glob

def load_markdown_files(folder_path: str) -> list[dict]:
    """
    Membaca semua file .md dari folder yang ditentukan.
    Returns list of dict: {filename, content}
    """
    md_files = glob.glob(os.path.join(folder_path, "**/*.md"), recursive=True)
    md_files += glob.glob(os.path.join(folder_path, "*.md"))
    md_files = list(set(md_files))  # deduplikasi

    if not md_files:
        raise FileNotFoundError(f"Tidak ada file .md ditemukan di folder: {folder_path}")

    documents = []
    for filepath in sorted(md_files):
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read().strip()
        if content:
            documents.append({
                "filename": os.path.basename(filepath),
                "filepath": filepath,
                "content": content
            })
            print(f"   📄 {os.path.basename(filepath)} ({len(content):,} karakter)")

    print(f"\n✅ Total {len(documents)} file .md berhasil dibaca!")
    return documents

documents = load_markdown_files(MD_FOLDER_PATH)

   📄 Pedoman_Penulisan_Skripsi_Fasilkom-TI_hanya gambar.md (14,206 karakter)
   📄 pedoman_tambahan.md (25,807 karakter)
   📄 pedoman_tulisan.md (61,483 karakter)
   📄 prosedur.md (10,154 karakter)

✅ Total 4 file .md berhasil dibaca!


# 3. Semantic Chunking

In [13]:
from semantic_text_splitter import TextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np
import re

def semantic_chunk_text(
    text: str,
    embed_model: SentenceTransformer,
    window_size: int = 2,
    percentile_threshold: int = 80,
    min_sentences: int = 10
) -> list[str]:
    """
    Melakukan semantic chunking dengan pendekatan percentile breakpoint.

    Algoritma:
    1. Pisahkan teks menjadi kalimat.
    2. Gabungkan kalimat dalam window (sliding window) → group.
    3. Hitung cosine similarity antar group yang berdekatan.
    4. Hitung breakpoint pada persentil ke-N dari distribusi similarity.
    5. Potong teks di titik similarity yang rendah (di bawah threshold).
    6. Gabungkan chunk yang terlalu kecil (< min_sentences) ke chunk berikutnya.
    """
    # Tokenisasi kalimat sederhana
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sentences) <= window_size:
        return [text]  # Tidak cukup kalimat untuk di-chunk

    # Buat window groups
    groups = []
    for i in range(len(sentences)):
        start = max(0, i - window_size // 2)
        end   = min(len(sentences), i + window_size // 2 + 1)
        groups.append(" ".join(sentences[start:end]))

    # Embedding semua groups
    embeddings = embed_model.encode(groups, show_progress_bar=False, normalize_embeddings=True)

    # Cosine similarity antar group berurutan
    similarities = [
        float(np.dot(embeddings[i], embeddings[i + 1]))
        for i in range(len(embeddings) - 1)
    ]

    # Hitung breakpoint (titik dengan similarity RENDAH = topik berganti)
    distances = [1 - s for s in similarities]
    breakpoint_val = np.percentile(distances, percentile_threshold)

    # Temukan indeks breakpoint
    break_indices = [i + 1 for i, d in enumerate(distances) if d >= breakpoint_val]

    # Potong kalimat berdasarkan breakpoint → hasilkan segmen awal
    segments = []   # list of list[str] (kumpulan kalimat per segmen)
    prev_idx = 0
    for idx in break_indices:
        seg = sentences[prev_idx:idx]
        if seg:
            segments.append(seg)
        prev_idx = idx
    remaining = sentences[prev_idx:]
    if remaining:
        segments.append(remaining)

    if not segments:
        return [text]

    # ── Enforce min_sentences: gabung chunk kecil ke chunk berikutnya ────────
    merged: list[list[str]] = []
    buffer: list[str] = []

    for seg in segments:
        buffer.extend(seg)
        if len(buffer) >= min_sentences:
            merged.append(buffer)
            buffer = []

    # Sisa buffer: jika cukup besar jadikan chunk sendiri,
    # jika tidak gabungkan ke chunk terakhir
    if buffer:
        if merged and len(buffer) < min_sentences:
            merged[-1].extend(buffer)   # gabung ke chunk terakhir
        else:
            merged.append(buffer)       # jadikan chunk baru

    # Konversi list kalimat → string
    chunks = [" ".join(seg).strip() for seg in merged if seg]
    return chunks if chunks else [text]


# ─── Load embedding model ─────────────────────────────────────────────────────
print(f"⏳ Memuat embedding model: {EMBEDDING_MODEL} ...")
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"✅ Model berhasil dimuat! Dimensi vektor: {embed_model.get_sentence_embedding_dimension()}")


# ─── Lakukan semantic chunking untuk semua dokumen ───────────────────────────
all_chunks = []  # List of dict: {chunk_id, text, source_file}

for doc in documents:
    chunks = semantic_chunk_text(
        text=doc["content"],
        embed_model=embed_model,
        window_size=WINDOW_SIZE,
        percentile_threshold=PERCENTILE_THRESH,
        min_sentences=MIN_SENTENCES
    )
    for i, chunk in enumerate(chunks):
        sent_count = len([s for s in re.split(r'(?<=[.!?])\s+', chunk) if s.strip()])
        all_chunks.append({
            "chunk_id"       : f"{doc['filename']}_chunk_{i}",
            "text"           : chunk,
            "source"         : doc["filename"],
            "filepath"       : doc["filepath"],
            "sentence_count" : sent_count
        })
    print(f"   ✂️  {doc['filename']}: {len(chunks)} chunks "
          f"(min {min(len([s for s in re.split(r'(?<=[.!?])\s+', c) if s.strip()]) for c in chunks)} kalimat/chunk)")

print(f"\n✅ Total chunks: {len(all_chunks)}")
if all_chunks:
    counts = [c['sentence_count'] for c in all_chunks]
    print(f"   Rata-rata kalimat/chunk : {sum(counts)/len(counts):.1f}")
    print(f"   Min kalimat/chunk       : {min(counts)}")
    print(f"   Max kalimat/chunk       : {max(counts)}")
print(f"\n📋 Contoh chunk pertama:")
print("-" * 60)
print(all_chunks[0]['text'][:400] if all_chunks else '(kosong)')


⏳ Memuat embedding model: paraphrase-multilingual-MiniLM-L12-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_118/3996098966.py:93: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Model berhasil dimuat! Dimensi vektor: {embed_model.get_sentence_embedding_dimension()}")


✅ Model berhasil dimuat! Dimensi vektor: 384
   ✂️  Pedoman_Penulisan_Skripsi_Fasilkom-TI_hanya gambar.md: 12 chunks (min 2 kalimat/chunk)
   ✂️  pedoman_tambahan.md: 50 chunks (min 2 kalimat/chunk)
   ✂️  pedoman_tulisan.md: 116 chunks (min 2 kalimat/chunk)
   ✂️  prosedur.md: 13 chunks (min 2 kalimat/chunk)

✅ Total chunks: 191
   Rata-rata kalimat/chunk : 5.8
   Min kalimat/chunk       : 2
   Max kalimat/chunk       : 44

📋 Contoh chunk pertama:
------------------------------------------------------------
# Pedoman Penulisan Skripsi
**Fakultas Ilmu Komputer dan Teknologi Informasi — Universitas Sumatera Utara**

---

> **Catatan:** Dokumen ini diekstrak secara semantik dari file asli yang seluruh kontennya berupa gambar. Semua ukuran, tata letak, dan teks telah direkonstruksi dari informasi visual dalam file tersebut. ---

## Daftar Lampiran

| Lampiran | Keterangan |
|----------|------------|
| 1A


# 4. Vector Embedding

In [14]:
from tqdm.auto import tqdm

print(f"⏳ Melakukan embedding {len(all_chunks)} chunks ...")

texts = [c["text"] for c in all_chunks]

embeddings = embed_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # Cosine similarity → dot product
)

print(f"\n✅ Embedding selesai!")
print(f"   Shape: {embeddings.shape}")
print(f"   Dimensi: {embeddings.shape[1]}")

⏳ Melakukan embedding 191 chunks ...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


✅ Embedding selesai!
   Shape: (191, 384)
   Dimensi: 384


# 5. Simpan Vektor Embedding ke Qdrant (Server Mode)

In [15]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams,
    PointStruct, PayloadSchemaType
)

# ─── Koneksi ke Qdrant Server ─────────────────────────────────────────────────
print(f"⏳ Menghubungkan ke Qdrant di {QDRANT_URL} ...")
qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60
)
print("✅ Koneksi berhasil!")


# ─── Buat atau reset collection ───────────────────────────────────────────────
existing_collections = [c.name for c in qdrant.get_collections().collections]

if COLLECTION_NAME in existing_collections:
    print(f"⚠️  Collection '{COLLECTION_NAME}' sudah ada, menghapus dan membuat ulang ...")
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_DIM,
        distance=Distance.COSINE
    )
)
print(f"✅ Collection '{COLLECTION_NAME}' berhasil dibuat!")


# ─── Upsert semua vektor ──────────────────────────────────────────────────────
BATCH_SIZE = 100

points = [
    PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload={
            "chunk_id" : all_chunks[idx]["chunk_id"],
            "text"     : all_chunks[idx]["text"],
            "source"   : all_chunks[idx]["source"],
            "filepath" : all_chunks[idx]["filepath"]
        }
    )
    for idx in range(len(all_chunks))
]

print(f"\n⏳ Menyimpan {len(points)} vektor ke Qdrant (batch size={BATCH_SIZE}) ...")

for i in tqdm(range(0, len(points), BATCH_SIZE), desc="Upload"):
    batch = points[i:i + BATCH_SIZE]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=batch)

# Verifikasi
count = qdrant.count(collection_name=COLLECTION_NAME).count
print(f"\n✅ Upload selesai! Total vektor tersimpan: {count}")

⏳ Menghubungkan ke Qdrant di https://ea295c4d-3efb-4676-9964-a20d09b4e162.us-east4-0.gcp.cloud.qdrant.io ...
✅ Koneksi berhasil!
⚠️  Collection 'rag_docs' sudah ada, menghapus dan membuat ulang ...
✅ Collection 'rag_docs' berhasil dibuat!

⏳ Menyimpan 191 vektor ke Qdrant (batch size=100) ...


Upload:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Upload selesai! Total vektor tersimpan: 191


# 6. Input Query dari User

In [16]:
# ─── Masukkan pertanyaan di sini ──────────────────────────────────────────────
user_query = input("🔍 Masukkan pertanyaan Anda: ").strip()

if not user_query:
    raise ValueError("❌ Query tidak boleh kosong!")

print(f"\n📝 Query: {user_query}")

🔍 Masukkan pertanyaan Anda:  Apa langkah-langkah maju seminar proposal?



📝 Query: Apa langkah-langkah maju seminar proposal?


# 7. Embedding Query User

In [17]:
print("⏳ Melakukan embedding query ...")

query_embedding = embed_model.encode(
    user_query,
    normalize_embeddings=True
).tolist()

print(f"✅ Query berhasil di-embed! Dimensi: {len(query_embedding)}")

⏳ Melakukan embedding query ...
✅ Query berhasil di-embed! Dimensi: 384


# 8. Retrieval dari Qdrant (Top-K = 25)

In [18]:
print(f"⏳ Melakukan retrieval top-{TOP_K} dari Qdrant ...")

# qdrant-client v2.x: gunakan query_points (pengganti .search yang deprecated)
search_results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=TOP_K,
    with_payload=True
).points

retrieved_chunks = [
    {
        "id"     : hit.id,
        "text"   : hit.payload["text"],
        "source" : hit.payload["source"],
        "score"  : hit.score
    }
    for hit in search_results
]

print(f"✅ Berhasil retrieve {len(retrieved_chunks)} chunks!")
print("\n📋 Top 5 hasil retrieval (sebelum reranking):")
print("-" * 60)
for i, chunk in enumerate(retrieved_chunks[:5], 1):
    print(f"[{i}] Score: {chunk['score']:.4f} | Sumber: {chunk['source']}")
    print(f"    {chunk['text'][:150]}...\n")

⏳ Melakukan retrieval top-25 dari Qdrant ...
✅ Berhasil retrieve 25 chunks!

📋 Top 5 hasil retrieval (sebelum reranking):
------------------------------------------------------------
[1] Score: 0.7561 | Sumber: pedoman_tambahan.md
    4. Tahap Pelaksanaan Seminar Proposal
• Penjadwalan: Pihak Prodi / Fakultas melakukan Penjadwalan Seminar Proposal. • Pelaksanaan: Prodi / Fakultas me...

[2] Score: 0.7269 | Sumber: pedoman_tambahan.md
    6. Telah melakukan bimbingan proposal dengan kedua dosen pembimbing dan proposal
sudah di-acc oleh kedua dosen pembimbing. b. Seminar Proposal
Pada pe...

[3] Score: 0.6586 | Sumber: pedoman_tambahan.md
    3. Mahasiswa wajib menyiapkan diri untuk memberikan penjelasan mengenai proposal yang
diajukan pada seminar proposal. 4. Mahasiswa akan memaparkan ren...

[4] Score: 0.6533 | Sumber: prosedur.md
    Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer

Diperuntukkan Bagi Mahasiswa Program Studi Ilmu Komputer yang sedang mengambil Mata
Kuliah Tugas Akhir. ..

# 9. Reranking dengan CrossEncoder (Top-5)

In [19]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"

# ─── Inisialisasi CrossEncoder Reranker ──────────────────────────────────────
print(f"⏳ Memuat CrossEncoder reranker: {RERANKER_MODEL} ...")
cross_encoder = CrossEncoder(
    model_name=RERANKER_MODEL,
    max_length=512
)
print("✅ CrossEncoder reranker siap!")

The CrossEncoder `model_name` argument was renamed and is now deprecated. Please use `model_name_or_path` instead.


⏳ Memuat CrossEncoder reranker: cross-encoder/ms-marco-MiniLM-L-12-v2 ...


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ CrossEncoder reranker siap!


In [20]:
# ─── Buat pasangan (query, passage) untuk scoring ────────────────────────────
pairs = [[user_query, chunk["text"]] for chunk in retrieved_chunks]

# ─── Lakukan reranking ────────────────────────────────────────────────────────
print(f"\n⏳ Melakukan reranking {len(pairs)} chunks ...")
rerank_scores = cross_encoder.predict(pairs, show_progress_bar=True)

# Gabungkan score ke retrieved_chunks, lalu sort descending
for chunk, score in zip(retrieved_chunks, rerank_scores):
    chunk["rerank_score"] = float(score)

top_reranked = sorted(
    retrieved_chunks,
    key=lambda x: x["rerank_score"],
    reverse=True
)[:TOP_N_RERANK]

print(f"\n✅ Reranking selesai! Menampilkan top-{TOP_N_RERANK}:")
print("-" * 60)
for idx, result in enumerate(top_reranked, 1):
    print(f"[{idx}] Rerank Score : {result['rerank_score']:.4f} | Sumber: {result['source']}")
    print(f"     Vector Score: {result['score']:.4f}")
    print(f"     {result['text'][:150]}...\n")


⏳ Melakukan reranking 25 chunks ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Reranking selesai! Menampilkan top-5:
------------------------------------------------------------
[1] Rerank Score : 2.1155 | Sumber: prosedur.md
     Vector Score: 0.6527
     Penelitian
9. Exum (Judul, Latar Belakang, Rumusan Masalah, Metodologi, Referensi)
10. Tanda tangan mahasiswa

Seminar Proposal

Untuk  maju  seminar...

[2] Rerank Score : 1.0931 | Sumber: pedoman_tambahan.md
     Vector Score: 0.7269
     6. Telah melakukan bimbingan proposal dengan kedua dosen pembimbing dan proposal
sudah di-acc oleh kedua dosen pembimbing. b. Seminar Proposal
Pada pe...

[3] Rerank Score : 0.2948 | Sumber: pedoman_tambahan.md
     Vector Score: 0.7561
     4. Tahap Pelaksanaan Seminar Proposal
• Penjadwalan: Pihak Prodi / Fakultas melakukan Penjadwalan Seminar Proposal. • Pelaksanaan: Prodi / Fakultas me...

[4] Rerank Score : -0.4444 | Sumber: pedoman_tambahan.md
     Vector Score: 0.6586
     3. Mahasiswa wajib menyiapkan diri untuk memberikan penjelasan mengenai proposal yang
diajuk

# 10. Mengatur Final Prompt (Vectors + Query + Prompt Template)

In [21]:
# ─── Gabungkan context dari top reranked chunks ───────────────────────────────
context_parts = []
for idx, result in enumerate(top_reranked, 1):
    source = result["source"]
    text   = result["text"]
    context_parts.append(f"[Source {idx}: {source}]\n{text}")

context = "\n\n".join(context_parts)

# ─── Prompt Template ─────────────────────────────────────────────────────────
PROMPT_TEMPLATE = """Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Query: {query}

Answer the question and provide additional helpful information,
based on the pieces of information, if applicable. Be succinct.
Responses should be properly formatted to be easily read."""

final_prompt = PROMPT_TEMPLATE.format(
    context=context,
    query=user_query
)

print("✅ Prompt berhasil dibuat!")
print(f"   Total karakter prompt: {len(final_prompt):,}")
print("\n" + "=" * 60)
print("PREVIEW PROMPT:")
print("=" * 60)
print(final_prompt[:800] + "\n...[dipotong untuk preview]" if len(final_prompt) > 800 else final_prompt)

✅ Prompt berhasil dibuat!
   Total karakter prompt: 4,903

PREVIEW PROMPT:
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
[Source 1: prosedur.md]
Penelitian
9. Exum (Judul, Latar Belakang, Rumusan Masalah, Metodologi, Referensi)
10. Tanda tangan mahasiswa

Seminar Proposal

Untuk  maju  seminar  proposal  mahasiswa  perlu  menyelesaikan  dua  submission  di  bawah  ini
secara berurutan. 1. Persetujuan Seminar Proposal

Mahasiswa  diharuskan  mendapatkan  persetujuan  untuk  mengikuti  seminar  Proposal  oleh
dosen pembimbing melalui modul ini. Mahasiswa wajib mengirimkan file submission berupa :

1. File Proposal dengan format :  PROPOSAL_NIM.pdf
2. File  Form  Persetujuan  Seminar  Proposal  yang  sudah  diisi  dan  ditandatangani  oleh

...[dipotong untuk preview]


# 11. Generate Hasil Menggunakan LLM (Llama 4)

In [40]:
from groq import Groq

# ─── Inisialisasi Groq client ─────────────────────────────────────────────────
groq_client = Groq(api_key=GROQ_API_KEY)

print(f"⏳ Mengirim prompt ke Groq ({GROQ_MODEL}) ...")

chat_completion = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {
            "role"   : "system",
            "content": "You are a helpful assistant that answers questions based on provided context."
        },
        {
            "role"   : "user",
            "content": final_prompt
        }
    ],
    temperature=0.2,      # Lebih deterministik untuk RAG
    max_tokens=1024,
    top_p=0.9
)

llm_answer  = chat_completion.choices[0].message.content
usage       = chat_completion.usage

print("✅ Respons diterima dari Groq!")
print(f"   Prompt tokens     : {usage.prompt_tokens:,}")
print(f"   Completion tokens : {usage.completion_tokens:,}")
print(f"   Total tokens      : {usage.total_tokens:,}")

from IPython.display import Markdown, display

print("=" * 70)
print("🔍 PERTANYAAN:")
print("=" * 70)
print(user_query)

print("\n" + "=" * 70)
print(f"🤖 JAWABAN  ({GROQ_MODEL}):")
print("=" * 70)

display(Markdown(llm_answer))

⏳ Mengirim prompt ke Groq (meta-llama/llama-4-scout-17b-16e-instruct) ...
✅ Respons diterima dari Groq!
   Prompt tokens     : 1,199
   Completion tokens : 231
   Total tokens      : 1,430
🔍 PERTANYAAN:
Apa langkah-langkah maju seminar proposal?

🤖 JAWABAN  (meta-llama/llama-4-scout-17b-16e-instruct):


**Langkah-Langkah Maju Seminar Proposal**

Berikut adalah langkah-langkah untuk maju seminar proposal:

1. **Mendapatkan Persetujuan Seminar Proposal**:
 * Mengirimkan file proposal dengan format `PROPOSAL_NIM.pdf`.
 * Mengirimkan file Form Persetujuan Seminar Proposal yang sudah diisi dan ditandatangani oleh Dosen Pembimbing dengan format `DOPING_NIM.pdf` beserta dengan lembar kendali bimbingan pra proposal yang telah ditandatangani oleh dosen pembimbing.
2. **Melakukan Bimbingan Proposal**:
 * Telah melakukan bimbingan proposal dengan kedua dosen pembimbing.
 * Proposal sudah di-acc oleh kedua dosen pembimbing.
3. **Mengikuti Workshop Seminar Proposal**:
 * Melakukan submission kembali dengan proposal yang sudah siap dengan format `Proposal_NIM.pdf`.

**Informasi Tambahan**

* Seminar Proposal dapat dilaksanakan secara daring ataupun luring.
* Mahasiswa wajib menyiapkan diri untuk memberikan penjelasan mengenai proposal yang diajukan pada seminar proposal.
* Dalam seminar proposal, akan diputuskan apakah judul dan isi Tugas Akhir/Skripsi disetujui atau tidak, dan apakah perlu perbaikan proposal dilakukan.

# 12. Mode Interaktif — Tanya Berulang

In [43]:
def ask(query: str) -> str:
    """Fungsi lengkap: query → embed → retrieve → rerank → generate → return answer."""

    # Embed query
    q_embedding = embed_model.encode(query, normalize_embeddings=True).tolist()

    # Retrieve top-K dari Qdrant
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding,
        limit=TOP_K,
        with_payload=True
    ).points

    candidates = [
        {
            "id"     : h.id,
            "text"   : h.payload["text"],
            "source" : h.payload["source"],
            "score"  : h.score
        }
        for h in hits
    ]

    # CrossEncoder reranking
    pairs  = [[query, c["text"]] for c in candidates]
    scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)

    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:10]

    # Build context
    context = "\n\n".join(
        f"[Source {i+1}: {r['source']}]\n{r['text']}"
        for i, r in enumerate(reranked)
    )

    # Build prompt
    prompt = PROMPT_TEMPLATE.format(context=context, query=query)

    # Call Groq
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers questions based on provided context."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.2,
        max_tokens=2048
    )
    return resp.choices[0].message.content


# ─── Loop interaktif ──────────────────────────────────────────────────────────
print("💬 Mode Interaktif RAG — ketik 'keluar' untuk berhenti\n")
while True:
    q = input("🔍 Pertanyaan: ").strip()
    if q.lower() in ("keluar", "exit", "quit", "q"):
        print("👋 Sesi selesai.")
        break
    if not q:
        continue
    print("\n⏳ Memproses ...\n")
    answer = ask(q)
    from IPython.display import Markdown, display
    
    print("=" * 70)
    print("🔍 PERTANYAAN:")
    print("=" * 70)
    print(q)
    
    print("\n" + "=" * 70)
    print(f"🤖 JAWABAN  ({GROQ_MODEL}):")
    print("=" * 70)
    
    display(Markdown(answer))

💬 Mode Interaktif RAG — ketik 'keluar' untuk berhenti



🔍 Pertanyaan:  apa saja dokumen-dokumen untuk maju seminar hasil?



⏳ Memproses ...

🔍 PERTANYAAN:
apa saja dokumen-dokumen untuk maju seminar hasil?

🤖 JAWABAN  (meta-llama/llama-4-scout-17b-16e-instruct):


## Dokumen-dokumen untuk Maju Seminar Hasil

Berikut adalah dokumen-dokumen yang diperlukan untuk maju seminar hasil:

1. **File Skripsi Final (Full)**: dengan format `NIM_SKRIPSI_CHECK.docx` (File word) untuk plagiarism check.
2. **Form Persetujuan Seminar Hasil**: yang telah diisi dan ditandatangani oleh Dosen Pembimbing 1 dan 2, dengan format `PERSETUJUAN_NIM.pdf`.
3. **Form lembar kendali bimbingan pra seminar hasil**: yang telah ditandatangani oleh Dosen Pembimbing 1 dan 2.
4. **Berkas Berita Acara Seminar Hasil**: dengan format `BASemHas_NIM.doc/docx` yang telah terisi kecuali pada bagian nilai.

Pastikan Anda telah memenuhi persyaratan lainnya, seperti telah mengambil MK TA di KRS pada semester berjalan, dan telah melalui tahap-tahap sebelumnya dengan baik.

🔍 Pertanyaan:  Apa saja persyaratan seminar proposal?



⏳ Memproses ...

🔍 PERTANYAAN:
Apa saja persyaratan seminar proposal?

🤖 JAWABAN  (meta-llama/llama-4-scout-17b-16e-instruct):


**Persyaratan Seminar Proposal**

Berdasarkan informasi yang tersedia, persyaratan seminar proposal adalah sebagai berikut:

1. **Proposal penelitian dapat disusun oleh mahasiswa setelah exum di-acc oleh kedua pembimbing** (Source 2).
2. **Mahasiswa harus mendapatkan persetujuan untuk mengikuti seminar proposal oleh dosen pembimbing** (Source 4).
3. **Mahasiswa wajib mengirimkan file submission berupa**:
 * File Proposal dengan format: `PROPOSAL_NIM.pdf`
 * File Form Persetujuan Seminar Proposal yang sudah diisi dan ditandatangani oleh Dosen Pembimbing dengan format: `DOPING_NIM.pdf` beserta dengan lembar kendali bimbingan pra proposal yang telah ditandatangani oleh dosen pembimbing (Source 4).
4. **Telah melakukan bimbingan proposal dengan kedua dosen pembimbing dan proposal sudah di-acc oleh kedua dosen pembimbing** (Source 7).

**Informasi Tambahan**

* Mahasiswa dapat mengajukan proposal Tugas Akhir walaupun tidak mengambil TA di KRS namun tetap mengikuti persyaratan lainnya yang telah ditentukan (Source 2).
* Mahasiswa yang sedang MBKM juga diperbolehkan mengajukan proposal Tugas Akhir dengan pertimbangan tidak mengganggu kegiatan MBKM (Source 2).

🔍 Pertanyaan:  exit


👋 Sesi selesai.
